# Feature Analysis v2 — CSIC 2010

**Versión:** v2  
**Fecha:** 2026-04-11  
**Notebook anterior:** `csic2010_feature_analysis_v1.ipynb`  
**Preprocessing:** `preprocess_csic_v2.py` → `features_v2.parquet` (17 features + label)

## Qué cambió respecto a v1

v1 descubrió que los falsos positivos no tienen indicadores de payload — el modelo los clasifica como ataque solo por longitud de URL/content. Se agregaron dos features que dan contexto a esa longitud:

| Feature nueva | Descripción | Correlación con label (v1) |
|---|---|---|
| `url_pct_density` | Ratio de chars encoded (%XX) sobre longitud total de URL | 0.267 |
| `url_param_count` | Cantidad de parámetros en la URL (conteo de `=`) | 0.146 |

## Objetivo

Re-entrenar los 4 modelos (LR, RF, XGBoost, LightGBM) con las 17 features nuevas y verificar si la Precision mejora respecto a v1 manteniendo Recall ≥ 0.95.

## Resultado esperado

En v1, el RF extendido con estas 2 features mostró Precision 0.704 vs 0.655 del RF base (+0.049). Este notebook confirma ese resultado con todos los modelos y con el dataset procesado oficial.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve,
    recall_score, precision_score,
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

pd.set_option('display.max_columns', None)

# v2 usa features_v2.parquet
DATA_PATH    = '../../data/processed/csic2010/features_v2.parquet'
RANDOM_STATE = 42
MIN_RECALL   = 0.95
MIN_PREC     = 0.85

# Resultados de v1 para comparar al final
V1_RESULTS = {
    'Logistic Regression': {'roc_auc': 0.761, 'recall': 1.000, 'precision': 0.411},
    'Random Forest':       {'roc_auc': 0.939, 'recall': 0.951, 'precision': 0.655},
    'XGBoost':             {'roc_auc': 0.933, 'recall': 0.964, 'precision': 0.594},
    'LightGBM':            {'roc_auc': 0.941, 'recall': 0.953, 'precision': 0.654},
}

## 1. Cargar datos

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Features: {df.drop(columns=["label"]).columns.tolist()}')
print(f'\nLabel:\n{df["label"].value_counts()}')

feature_names = df.drop(columns=['label']).columns.tolist()
X = df.drop(columns=['label']).values.astype('float32')
y = df['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)

# Escalar features continuas — fit solo en train
continuous_cols = ['url_length', 'url_param_count', 'url_pct_density', 'content_length']
continuous_idx  = [feature_names.index(c) for c in continuous_cols]
scaler = StandardScaler()
X_train[:, continuous_idx] = scaler.fit_transform(X_train[:, continuous_idx])
X_val[:, continuous_idx]   = scaler.transform(X_val[:, continuous_idx])
X_test[:, continuous_idx]  = scaler.transform(X_test[:, continuous_idx])

print(f'\nTrain: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')

Shape: (61065, 18)
Features: ['method_is_get', 'method_is_post', 'method_is_put', 'url_length', 'url_param_count', 'url_pct_density', 'url_has_pct27', 'url_has_pct3c', 'url_has_dashdash', 'url_has_script', 'url_has_select', 'content_length', 'content_has_pct27', 'content_has_pct3c', 'content_has_dashdash', 'content_has_script', 'content_has_select']

Label:
label
0    36000
1    25065
Name: count, dtype: int64

Train: 42745 | Val: 9160 | Test: 9160


## 2. Entrenar y evaluar los 4 modelos

In [3]:
def find_best_threshold(y_true, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    mask = recalls[:-1] >= MIN_RECALL
    if not mask.any():
        return float(thresholds[np.argmax(recalls[:-1])])
    return float(thresholds[np.where(mask, precisions[:-1], 0).argmax()])

def run_model(name, model):
    model.fit(X_train, y_train)
    val_proba  = model.predict_proba(X_val)[:, 1]
    threshold  = find_best_threshold(y_val, val_proba)
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred  = (test_proba >= threshold).astype(int)
    
    roc  = roc_auc_score(y_test, test_proba)
    rec  = recall_score(y_test, test_pred)
    prec = precision_score(y_test, test_pred)
    cm   = confusion_matrix(y_test, test_pred)
    
    # Comparar con v1
    v1 = V1_RESULTS.get(name, {})
    delta_roc  = roc  - v1.get('roc_auc', 0)
    delta_prec = prec - v1.get('precision', 0)
    delta_rec  = rec  - v1.get('recall', 0)
    
    print(f'\n{"="*55}')
    print(f'Modelo: {name}')
    print(f'{"="*55}')
    print(f'Threshold:  {threshold:.4f}')
    print(f'ROC-AUC:    {roc:.4f}  (v1: {v1.get("roc_auc","-"):.3f}  Δ{delta_roc:+.3f})')
    print(f'Recall:     {rec:.4f}  (v1: {v1.get("recall","-"):.3f}  Δ{delta_rec:+.3f})  {"✅" if rec >= MIN_RECALL else "❌"}')
    print(f'Precision:  {prec:.4f}  (v1: {v1.get("precision","-"):.3f}  Δ{delta_prec:+.3f})  {"✅" if prec >= MIN_PREC else "❌"}')
    print(f'Confusion matrix:')
    print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
    print(f'  FN={cm[1,0]}  TP={cm[1,1]}')
    
    return {'name': name, 'roc_auc': roc, 'recall': rec, 'precision': prec, 'threshold': threshold}

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale = neg / pos

results = []
results.append(run_model('Logistic Regression',
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)))

results.append(run_model('Random Forest',
    RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)))

results.append(run_model('XGBoost',
    XGBClassifier(n_estimators=200, scale_pos_weight=scale, eval_metric='logloss',
                  random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)))

results.append(run_model('LightGBM',
    LGBMClassifier(n_estimators=200, scale_pos_weight=scale, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)))


Modelo: Logistic Regression
Threshold:  0.3105
ROC-AUC:    0.7665  (v1: 0.761  Δ+0.006)
Recall:     0.9577  (v1: 1.000  Δ-0.042)  ✅
Precision:  0.4201  (v1: 0.411  Δ+0.009)  ❌
Confusion matrix:
  TN=429  FP=4971
  FN=159  TP=3601

Modelo: Random Forest
Threshold:  0.1445
ROC-AUC:    0.9497  (v1: 0.939  Δ+0.011)
Recall:     0.9503  (v1: 0.951  Δ-0.001)  ✅
Precision:  0.7038  (v1: 0.655  Δ+0.049)  ❌
Confusion matrix:
  TN=3896  FP=1504
  FN=187  TP=3573

Modelo: XGBoost
Threshold:  0.1173
ROC-AUC:    0.9450  (v1: 0.933  Δ+0.012)
Recall:     0.9630  (v1: 0.964  Δ-0.001)  ✅
Precision:  0.6329  (v1: 0.594  Δ+0.039)  ❌
Confusion matrix:
  TN=3300  FP=2100
  FN=139  TP=3621

Modelo: LightGBM
Threshold:  0.1422
ROC-AUC:    0.9525  (v1: 0.941  Δ+0.012)
Recall:     0.9532  (v1: 0.953  Δ+0.000)  ✅
Precision:  0.7018  (v1: 0.654  Δ+0.048)  ❌
Confusion matrix:
  TN=3877  FP=1523
  FN=176  TP=3584


/Users/permotion/Desktop/repositories/PERMOTION/PMT MLSec/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/permotion/Desktop/repositories/PERMOTION/PMT MLSec/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 3. Tabla comparativa v1 vs v2

In [ ]:
print(f'{"-"*70}')
print(f'{"Modelo":<25} {"ROC-AUC":>10} {"Recall":>10} {"Precision":>10} {"Estado":>8}')
print(f'{"-"*70}')
for r in results:
    v1 = V1_RESULTS[r['name']]
    estado = '✅' if r['recall'] >= MIN_RECALL and r['precision'] >= MIN_PREC else '❌'
    print(f'{r["name"]:<25} {r["roc_auc"]:>10.4f} {r["recall"]:>10.4f} {r["precision"]:>10.4f} {estado:>8}')
    print(f'{"  vs v1":<25} {v1["roc_auc"]:>10.3f} {v1["recall"]:>10.3f} {v1["precision"]:>10.3f}')
    print()
print(f'{"-"*70}')
print(f'Criterios: Recall >= {MIN_RECALL} | Precision >= {MIN_PREC}')

## 4. Feature importance — RF v2

In [ ]:
# Entrenar RF de nuevo para acceder a feature_importances_
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
importances.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Feature importance — Random Forest v2 (17 features)')
ax.set_ylabel('Importancia')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nRanking completo:')
print(importances.round(4).to_string())

## 5. Observaciones

> Completar después de ejecutar

- **Mejora de Precision respecto a v1:** _ver tabla sección 3_
- **Posición de url_pct_density en feature importance:** _ver sección 4_
- **Posición de url_param_count en feature importance:** _ver sección 4_
- **¿Se cumplen los criterios de éxito?** Recall ≥ 0.95 / Precision ≥ 0.85
- **Decisión:** _continuar con v3 o pasar a Modelo B_